In [ ]:
%matplotlib inline

In [1]:
#Import your libraries here

import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

In [2]:
#Import your modules here
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utils.ad_parsing import ad_parsing_utils as adp
from utils.fetch_data import (
                                ScrapeSelection,
                                collect_listing_urls_for_routes_parallel_streaming,
                                download_and_parse_listing_batch_streaming, 
                                load_valid_deal_types,
                                load_valid_geo_paths,
                                load_valid_property_types
                                )

In [14]:
RAW_HTML_DIR = PROJECT_ROOT / "data" / "raw_listing_html"
RAW_HTML_DIR.mkdir(parents=True, exist_ok=True)

for file_path in RAW_HTML_DIR.glob("*.html"):
    file_path.unlink()

TAXONOMY_DIR = PROJECT_ROOT / "data" / "taxonomy"
DATA_DIR = PROJECT_ROOT / "data"

## Call the generated url and obtain a list of urls with ads to the related query

In [4]:
valid_deal_types = load_valid_deal_types(TAXONOMY_DIR / "valid_deal_types.csv")
valid_geo_paths = load_valid_geo_paths(TAXONOMY_DIR / "valid_geo_paths.csv")
valid_property_types = load_valid_property_types(TAXONOMY_DIR / "valid_property_types.csv")


all_sales_geo_paths = [
    geo_path
    for deal_type, geo_path in valid_geo_paths
    if deal_type == "prodazhbi"
]

selection = ScrapeSelection(
    deal_types=["prodazhbi"],
    geo_paths=all_sales_geo_paths,
    property_types=list(valid_property_types),
)

route_urls = selection.build_urls(
    valid_deal_types,
    valid_geo_paths,
    valid_property_types,
)

print("Geo paths:", len(all_sales_geo_paths))
print("Property types:", len(valid_property_types))
print("Route URLs:", len(route_urls))
print(route_urls[:10])

run_summary = collect_listing_urls_for_routes_parallel_streaming(
    route_urls=route_urls,
    max_workers=12,
    max_pages=200,
    delay_seconds=0.8,
    checkpoint_dir=PROJECT_ROOT / "data" / "route_discovery_sales_full_streaming",
    progress_every=100,
)

run_summary

Geo paths: 4366
Property types: 22
Route URLs: 96052
['https://www.imot.bg/obiavi/prodazhbi/oblast-stara-zagora/s-koprinka/atelie-tavan', 'https://www.imot.bg/obiavi/prodazhbi/oblast-stara-zagora/s-koprinka/hotel', 'https://www.imot.bg/obiavi/prodazhbi/oblast-stara-zagora/s-koprinka/myasto', 'https://www.imot.bg/obiavi/prodazhbi/oblast-stara-zagora/s-koprinka/partsel', 'https://www.imot.bg/obiavi/prodazhbi/oblast-stara-zagora/s-koprinka/tristaen', 'https://www.imot.bg/obiavi/prodazhbi/oblast-stara-zagora/s-koprinka/sklad', 'https://www.imot.bg/obiavi/prodazhbi/oblast-stara-zagora/s-koprinka/mnogostaen', 'https://www.imot.bg/obiavi/prodazhbi/oblast-stara-zagora/s-koprinka/mezonet', 'https://www.imot.bg/obiavi/prodazhbi/oblast-stara-zagora/s-koprinka/etazh-ot-kashta', 'https://www.imot.bg/obiavi/prodazhbi/oblast-stara-zagora/s-koprinka/magazin']
Total routes: 96052
Already completed: 0
Pending routes: 96052
[1/96052] routes completed | new this run=1 | route listings=0 | pages=1 | stop=n

{'total_routes': 96052,
 'already_completed': 0,
 'newly_completed': 96052,
 'new_page_rows': 116554,
 'new_listing_rows': 271493,
 'route_results_path': WindowsPath('D:/users/kamen.dimitrov/desktop/SOFTUNI/AI_and_ML_upskill_program/real_estate_valuation/data/route_discovery_sales_full_streaming/route_results.csv'),
 'page_results_path': WindowsPath('D:/users/kamen.dimitrov/desktop/SOFTUNI/AI_and_ML_upskill_program/real_estate_valuation/data/route_discovery_sales_full_streaming/page_results.csv'),
 'listing_urls_path': WindowsPath('D:/users/kamen.dimitrov/desktop/SOFTUNI/AI_and_ML_upskill_program/real_estate_valuation/data/route_discovery_sales_full_streaming/listing_urls_raw.csv')}

In [5]:
listing_urls_raw_path = PROJECT_ROOT / "data" / "route_discovery_sales_full_streaming" / "listing_urls_raw.csv"
listing_urls_final_path = PROJECT_ROOT / "data" / "route_discovery_sales_full_streaming" / "listing_urls_unique.csv"

listing_urls_df = pd.read_csv(listing_urls_raw_path)

unique_listing_urls_df = (
    listing_urls_df[["listing_url"]]
    .drop_duplicates()
    .sort_values("listing_url")
    .reset_index(drop=True)
)

unique_listing_urls_df.to_csv(listing_urls_final_path, index=False)

print("Raw listing rows:", len(listing_urls_df))
print("Unique listing URLs:", len(unique_listing_urls_df))
print("Saved to:", listing_urls_final_path)

Raw listing rows: 271493
Unique listing URLs: 166434
Saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\real_estate_valuation\data\route_discovery_sales_full_streaming\listing_urls_unique.csv


## Read the list of URLS from the generated query

In [3]:
# ============================================================
# Load discovered listing URLs from full geo/type route crawl
# Then download + parse listings using streaming pipeline
# ============================================================

# Where your 96k route crawl wrote its outputs
ROUTE_DISCOVERY_DIR = PROJECT_ROOT / "data" / "route_discovery_sales_full_streaming"

# This is created by collect_listing_urls_for_routes_parallel_streaming(...)
RAW_LISTING_URLS_CSV = ROUTE_DISCOVERY_DIR / "listing_urls_unique.csv"

# Where download + parse outputs should go
PARSED_OUTPUT_DIR = PROJECT_ROOT / "data" / "parsed_sales_full"


# ------------------------------------------------------------
# 1. Load listing URLs from route crawl output
# ------------------------------------------------------------

df_urls = pd.read_csv(RAW_LISTING_URLS_CSV)

listing_urls = (
    df_urls["listing_url"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Loaded unique listing URLs:", len(listing_urls))
print(listing_urls[:5])


# ------------------------------------------------------------
# 2. Download + parse listings using streaming/resumable method
# ------------------------------------------------------------

download_parse_summary = download_and_parse_listing_batch_streaming(
    listing_urls=listing_urls,
    output_dir=PARSED_OUTPUT_DIR,
    max_workers=24,
    limit=None,          # None = process all
    chunk_size=3000,     # keeps futures bounded
    progress_every=300,
)

download_parse_summary


# ------------------------------------------------------------
# 3. Optional: inspect parsed output
# ------------------------------------------------------------

parsed_csv = PARSED_OUTPUT_DIR / "parsed_listings.csv"
manifest_csv = PARSED_OUTPUT_DIR / "download_manifest.csv"

print("Download manifest:", manifest_csv)
print("Parsed listings:", parsed_csv)

if parsed_csv.exists():
    df_parsed = pd.read_csv(parsed_csv)
    print("Parsed rows:", len(df_parsed))
    display(df_parsed.head())
else:
    print("No parsed listings CSV created yet.")

Loaded unique listing URLs: 166434
['https://www.imot.bg/obiava-1a126881929164804-prodava-ednostaen-apartament-grad-pleven-druzhba-1-bivshiyat-bikarnik', 'https://www.imot.bg/obiava-1a127539284802892-prodava-ednostaen-apartament-oblast-dobrich-gr-balchik', 'https://www.imot.bg/obiava-1a129794411209548-prodava-ednostaen-apartament-oblast-blagoevgrad-gr-bansko', 'https://www.imot.bg/obiava-1a130799086731769-prodava-ednostaen-apartament-grad-pleven-druzhba-4', 'https://www.imot.bg/obiava-1a130892326438323-prodava-ednostaen-apartament-grad-sofiya-ovcha-kupel-1']
Total listing URLs: 166434
Already completed: 1532
Pending URLs: 164902
[1533/166434] listings processed | new this run=1 | status=200 | parsed=True | url=https://www.imot.bg/obiava-1a175690577875642-prodava-ednostaen-apartament-oblast-burgas-gr-tsarevo
[1832/166434] listings processed | new this run=300 | status=200 | parsed=True | url=https://www.imot.bg/obiava-1a175983426164967-prodava-ednostaen-apartament-grad-stara-zagora-tri-

C:\Users\kamen.dimitrov\AppData\Local\Temp\ipykernel_19032\2729563066.py:60: DtypeWarning: Columns (25,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df_parsed = pd.read_csv(parsed_csv)


Parsed rows: 164073


,ad_id,ad_url,ad_url_path,ad_url_slug,html_path,listing_type,title,deal_raw,property_type_raw,title_city_raw,...,description_raw,description_clean,features_raw,features_pipe,features_count,published_raw,views,training_eligible,parse_error,parsed_at
0,1a129794411209548,https://www.imot.bg/obiava-1a129794411209548-p...,/obiava-1a129794411209548-prodava-ednostaen-ap...,obiava-1a129794411209548-prodava-ednostaen-apa...,data\raw_listing_html\2caac228937b3b30.html,single_property_listing,"Продава 1-СТАЕН в област Благоевград, гр. Банс...",Продава,1-СТАЕН,област Благоевград,...,Предлагаме студио 32 кв.м в сграда Никмар на ...,Предлагаме студио 32 кв.м в сграда Никмар на ...,Тухла\nОбзаведен\nКонтрол на достъпа,Тухла|Обзаведен|Контрол на достъпа,3,NaN,5834.0,True,NaN,2026-05-16T13:50:20.885206+00:00
1,1a127539284802892,https://www.imot.bg/obiava-1a127539284802892-p...,/obiava-1a127539284802892-prodava-ednostaen-ap...,obiava-1a127539284802892-prodava-ednostaen-apa...,data\raw_listing_html\857c7fd39ad302de.html,single_property_listing,"Продава 1-СТАЕН в област Добрич, гр. Балчик - ...",Продава,1-СТАЕН,област Добрич,...,"Балчик, комплекс 'Марина Сити'- луксозни апарт...","Балчик, комплекс 'Марина Сити'- луксозни апарт...",Тухла\nАсансьор\nИнтернет връзка\nОбзаведен\nВ...,Тухла|Асансьор|Интернет връзка|Обзаведен|Видео...,8,NaN,2391.0,True,NaN,2026-05-16T13:50:20.903929+00:00
2,1a126881929164804,https://www.imot.bg/obiava-1a126881929164804-p...,/obiava-1a126881929164804-prodava-ednostaen-ap...,obiava-1a126881929164804-prodava-ednostaen-apa...,data\raw_listing_html\9208ce061d053c01.html,single_property_listing,"Продава 1-СТАЕН в град Плевен, Дружба 1 - 40 к...",Продава,1-СТАЕН,град Плевен,...,"Продава гарсониера след ремонт- стая, кухня, б...","Продава гарсониера след ремонт- стая, кухня, б...",Панел,Панел,1,NaN,6579.0,True,NaN,2026-05-16T13:50:21.030575+00:00
3,1a130799086731769,https://www.imot.bg/obiava-1a130799086731769-p...,/obiava-1a130799086731769-prodava-ednostaen-ap...,obiava-1a130799086731769-prodava-ednostaen-apa...,data\raw_listing_html\ad1172ee3bcfaaed.html,single_property_listing,"Продава 1-СТАЕН в град Плевен, Дружба 4 - 40 к...",Продава,1-СТАЕН,град Плевен,...,"Продава гарсониера след ремонт, с ПВС дограма,...","Продава гарсониера след ремонт, с ПВС дограма,...",Панел\nАсансьор,Панел|Асансьор,2,NaN,4282.0,True,NaN,2026-05-16T13:50:21.051889+00:00
4,1a140309247069043,https://www.imot.bg/obiava-1a140309247069043-p...,/obiava-1a140309247069043-prodava-ednostaen-ap...,obiava-1a140309247069043-prodava-ednostaen-apa...,data\raw_listing_html\733a85c00c43835a.html,single_property_listing,"Продава 1-СТАЕН в област Велико Търново, гр. Г...",Продава,1-СТАЕН,област Велико Търново,...,ПАНЕЛНА ГАСОНИЕРА---КУПУВАМЕ СПЕШНО.ПЛАЩТАНЕ В...,ПАНЕЛНА ГАСОНИЕРА---КУПУВАМЕ СПЕШНО.ПЛАЩТАНЕ В...,Панел\nС преход\nАсансьор\nС гараж\nС паркинг\...,Панел|С преход|Асансьор|С гараж|С паркинг|Лизи...,14,NaN,3.0,True,NaN,2026-05-16T13:50:24.307483+00:00


In [5]:
csv_path = PARSED_OUTPUT_DIR / "parsed_listings.csv"
parquet_path = PARSED_OUTPUT_DIR / "parsed_listings.parquet"

df = pd.read_csv(csv_path)

df.to_parquet(
    parquet_path,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print("Saved:", parquet_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

C:\Users\kamen.dimitrov\AppData\Local\Temp\ipykernel_19032\773456746.py:4: DtypeWarning: Columns (25,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Saved: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\real_estate_valuation\data\parsed_sales_full\parsed_listings.parquet
Rows: 164073
Columns: 40
